<a href="https://colab.research.google.com/github/SUZUB/ATU/blob/main/ATU_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# Cell 1: Infrastructure & Cloud Sync
# ==============================================================================
import os
from google.colab import drive

# 1. Public Git clone / pull
if not os.path.exists('ATU') and not os.path.exists('/content/ATU'):
    !git clone https://github.com/SUZUB/ATU.git
if os.path.exists('/content/ATU'):
    %cd /content/ATU
elif os.path.exists('ATU'):
    %cd ATU
!git pull origin main

# Install project requirements
!pip install -q -r requirements.txt

# 2. Google Drive Mount
drive.mount('/content/drive')

# 3. Persistent Drive Workspace & Subdirectory Creation
DRIVE_WORKSPACE = '/content/drive/MyDrive/ATU_Workspace'
os.makedirs(DRIVE_WORKSPACE, exist_ok=True)

DATASET_PATH = os.path.join(DRIVE_WORKSPACE, 'datasets')
MODEL_CHECKPOINTS = os.path.join(DRIVE_WORKSPACE, 'checkpoints')
RESULTS_PATH = os.path.join(DRIVE_WORKSPACE, 'results')

os.makedirs(DATASET_PATH, exist_ok=True)
os.makedirs(MODEL_CHECKPOINTS, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

print("\nInfrastructure & Cloud Sync complete.")
print(f"Project Directory: {os.getcwd()}")
print(f"Drive Workspace:   {DRIVE_WORKSPACE}")

In [ ]:
# ==============================================================================
# Cell 2: Imports & Setup
# ==============================================================================
# Standard library imports
import os
import sys
import json
import time
from pathlib import Path
from typing import Optional, Tuple, List, Dict, Any

# Third-party Data, ML, and RL Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import gymnasium as gym
from stable_baselines3 import PPO, A2C
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# Ensure ATU root directory is in sys.path
atu_root = Path.cwd()
if str(atu_root) not in sys.path:
    sys.path.insert(0, str(atu_root))

# Custom module imports
from market_data_processor import MarketDataProcessor, compute_split_index, create_final_state_vector
from sentiment_processor import SentimentProcessor
from constrained_trading_env import ConstrainedTradingEnv
from src.data_processing import load_archive_headlines
from src.train import calculate_financial_metrics

# Device Auto-detection
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Imports & Setup complete.")
print(f"PyTorch Version:  {torch.__version__}")
print(f"Active Device:    {DEVICE.upper()}")
if torch.cuda.is_available():
    print(f"GPU Hardware:     {torch.cuda.get_device_name(0)}")

In [ ]:
# ==============================================================================
# Cell 3: Real Market Data Ingestion & State Vector Merging (NO synthetic data)
# ==============================================================================
from datetime import time as dtime

TICKER = "AAPL"  # e.g. "RELIANCE.NS" for an NSE stock (matches the ₹ / MIS rules natively)
DRIVE_WORKSPACE = globals().get("DRIVE_WORKSPACE", "/content/drive/MyDrive/ATU_Workspace")
DRIVE_DATA_DIR = os.path.join(DRIVE_WORKSPACE, "datasets")

# Exchange session rules: square off 15 minutes before the close (NSE MIS = 15:15 IST)
if TICKER.upper().endswith((".NS", ".BO")):
    ENV_KWARGS = dict(session_start_time=dtime(9, 15), session_end_time=dtime(15, 30), auto_square_off_time=dtime(15, 15))
else:
    ENV_KWARGS = dict(session_start_time=dtime(9, 30), session_end_time=dtime(16, 0), auto_square_off_time=dtime(15, 45))

# 1. Real 5-minute OHLCV from Yahoo Finance (5m history is limited to the last 60 days).
#    Raises on failure: there is deliberately no random-walk fallback.
print(f"Downloading real 5-minute candles for {TICKER} (last 60 days)...")
market_processor = MarketDataProcessor()  # scaler is fit on the first 80% (train slice) only
market_df = market_processor.process(ticker=TICKER, period="60d", interval="5m")
bar_dates = pd.Series(market_df.index.date)
BARS_PER_DAY = int(bar_dates.value_counts().median())
market_start, market_end = bar_dates.min(), bar_dates.max()
print(f"Market Features Engineered: {market_df.shape[0]:,} candles x {market_df.shape[1]} columns")
print(f"Date Range: {market_start} -> {market_end} ({bar_dates.nunique()} sessions, ~{BARS_PER_DAY} bars/day, tz={market_df.index.tz})")

# 2. Real headlines from the archive, restricted to the market data window (no synthetic fallback)
try:
    headlines_df = load_archive_headlines(
        archive_dir=DRIVE_DATA_DIR,
        ticker=TICKER,
        max_rows=50000,
        allow_synthetic_fallback=False,
    )
except FileNotFoundError as e:
    print(f"No archive headlines available for {TICKER}: {str(e).splitlines()[0]}")
    headlines_df = pd.DataFrame(columns=["date", "headline"])

if not headlines_df.empty:
    headline_dates = pd.to_datetime(headlines_df["date"]).dt.date
    in_window = (headline_dates >= market_start - pd.Timedelta(days=7)) & (headline_dates <= market_end)
    print(f"Headlines loaded: {len(headlines_df):,} ({headline_dates.min()} -> {headline_dates.max()}); "
          f"{int(in_window.sum()):,} fall inside the market window")
    headlines_df = headlines_df[in_window.to_numpy()]

# 3. FinBERT sentiment only for headlines that overlap the price data
sentiment_df = None
if not headlines_df.empty:
    print(f"\nScoring {len(headlines_df):,} headlines via FinBERT on {DEVICE.upper()} (batch_size=32)...")
    sentiment_processor = SentimentProcessor(device=DEVICE)
    sentiment_df = sentiment_processor.process_news_dataframe(headlines_df, batch_size=32)
    print(f"FinBERT Sentiment Output: {len(sentiment_df):,} days scored")
else:
    print("\nWARNING: No real headlines overlap the market window -> daily_sentiment is neutral (0.0).")
    print("         The archive ends in 2020; add recent headlines to the datasets folder to enable sentiment.")

# 4. Merge (sentiment lagged by 1 day, stale scores expire after 4 days)
final_state_vector = create_final_state_vector(market_df, sentiment_df)
sentiment_series = final_state_vector["daily_sentiment"]
non_zero_pct = float((sentiment_series != 0.0).mean()) * 100.0

print(f"\nFinal State Vector Created successfully:")
print(f"  Shape:              {final_state_vector.shape}")
print(f"  Columns:            {list(final_state_vector.columns)}")
print(f"  Non-zero Sentiment: {non_zero_pct:.1f}% of state vectors")
print(f"  Env Session Rules:  {ENV_KWARGS}")
display(final_state_vector.head(3))

In [ ]:
# ==============================================================================
# Cell 4: Multi-Agent Ensemble RL (PPO + A2C) & Institutional Backtest
# ==============================================================================
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from stable_baselines3 import PPO, A2C
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

CHUNKS = 10
STEPS_PER_CHUNK = 50_000

# ------------------------------------------------------------------------------
# 1. Data Integration (Chronological 80% Train / 20% Test, same boundary the scaler was fit on)
# ------------------------------------------------------------------------------
assert "final_state_vector" in globals() and final_state_vector is not None, (
    "final_state_vector is missing from memory! Please execute Cell 3 first."
)

split_idx = compute_split_index(len(final_state_vector))
assert market_processor.fit_end_index == split_idx, "Scaler fit window does not match the train split!"
train_df = final_state_vector.iloc[:split_idx].copy()
test_df = final_state_vector.iloc[split_idx:].copy()

print(f"Data Integration (Real {TICKER} 5-min data):")
print(f"  Total Data:     {len(final_state_vector):,} candles")
print(f"  Training Set:   {len(train_df):,} candles ({len(set(train_df.index.date))} days)")
print(f"  Test Set:       {len(test_df):,} candles ({len(set(test_df.index.date))} days)")

# ------------------------------------------------------------------------------
# 2. Paths (persistent Google Drive workspace)
# ------------------------------------------------------------------------------
DRIVE_WORKSPACE = globals().get("DRIVE_WORKSPACE", "/content/drive/MyDrive/ATU_Workspace")
checkpoint_dir = os.path.join(DRIVE_WORKSPACE, "checkpoints")
tensorboard_dir = os.path.join(DRIVE_WORKSPACE, "training_logs")
RESULTS_PATH = os.path.join(DRIVE_WORKSPACE, "results")
for d in (checkpoint_dir, tensorboard_dir, RESULTS_PATH):
    os.makedirs(d, exist_ok=True)

ppo_checkpoint_path = os.path.join(checkpoint_dir, "ppo_trading_agent_v1.zip")
a2c_checkpoint_path = os.path.join(checkpoint_dir, "a2c_trading_agent_v1.zip")
ppo_vecnorm_path = os.path.join(checkpoint_dir, "ppo_vecnormalize.pkl")
a2c_vecnorm_path = os.path.join(checkpoint_dir, "a2c_vecnormalize.pkl")

# ------------------------------------------------------------------------------
# 3. Reward-normalized training environments (one per agent, stats persisted)
# ------------------------------------------------------------------------------
def make_train_env(stats_path: str) -> VecNormalize:
    """Raw rupee rewards are scaled by a running std for training only (clipped to +/-10)."""
    venv = DummyVecEnv([lambda: ConstrainedTradingEnv(df=train_df, **ENV_KWARGS)])
    if os.path.exists(stats_path):
        try:
            env = VecNormalize.load(stats_path, venv)
            env.training = True
            env.norm_reward = True
            print(f"  Restored reward-normalization stats: {stats_path}")
            return env
        except Exception as e:
            print(f"  Could not restore {stats_path} ({e}); using fresh stats.")
    return VecNormalize(venv, norm_obs=False, norm_reward=True, clip_reward=10.0, gamma=0.99)


def has_finite_weights(model) -> bool:
    return all(torch.isfinite(p).all().item() for p in model.policy.parameters())


def load_or_create(algo_cls, name: str, ckpt_path: str, env, **fresh_kwargs):
    """Resume from checkpoint when compatible and finite; otherwise start fresh (never deletes files)."""
    if os.path.exists(ckpt_path):
        try:
            model = algo_cls.load(ckpt_path, env=env, device="cpu")
            if not has_finite_weights(model):
                raise ValueError("checkpoint contains NaN/Inf weights")
            print(f"[Persistent Memory] Resumed {name} from {ckpt_path} ({model.num_timesteps:,} timesteps so far)")
            return model
        except Exception as e:
            print(f"[Fresh Initialization] Cannot resume {name} ({e}). Starting a fresh model.")
    else:
        print(f"[Fresh Initialization] No {name} checkpoint at {ckpt_path}.")
    return algo_cls(policy="MlpPolicy", env=env, verbose=1, seed=42, device="cpu", **fresh_kwargs)


print("\nBuilding training environments...")
ppo_train_env = make_train_env(ppo_vecnorm_path)
a2c_train_env = make_train_env(a2c_vecnorm_path)

ppo_model = load_or_create(
    PPO, "PPO", ppo_checkpoint_path, ppo_train_env,
    learning_rate=0.0003, n_steps=2048, batch_size=64, gamma=0.99, gae_lambda=0.95,
    clip_range=0.2, ent_coef=0.01, tensorboard_log=os.path.join(tensorboard_dir, "ppo"),
)
a2c_model = load_or_create(
    A2C, "A2C", a2c_checkpoint_path, a2c_train_env,
    learning_rate=0.0007, n_steps=5, gamma=0.99, gae_lambda=1.0, ent_coef=0.01,
    max_grad_norm=0.5, tensorboard_log=os.path.join(tensorboard_dir, "a2c"),
)

# ------------------------------------------------------------------------------
# 4. Chunked Training with Drive checkpointing (model + normalization stats)
# ------------------------------------------------------------------------------
for name, model, env, ckpt, stats in (
    ("PPO", ppo_model, ppo_train_env, ppo_checkpoint_path, ppo_vecnorm_path),
    ("A2C", a2c_model, a2c_train_env, a2c_checkpoint_path, a2c_vecnorm_path),
):
    print(f"\n[Training] {name}: {CHUNKS} chunks x {STEPS_PER_CHUNK:,} steps = {CHUNKS * STEPS_PER_CHUNK:,} timesteps")
    for chunk in range(CHUNKS):
        model.learn(total_timesteps=STEPS_PER_CHUNK, reset_num_timesteps=False, log_interval=1000)
        model.save(ckpt)
        env.save(stats)
        print(f"  {name} chunk {chunk + 1}/{CHUNKS} done ({model.num_timesteps:,} total timesteps) -> saved")
    if not has_finite_weights(model):
        raise RuntimeError(f"{name} weights became NaN/Inf during training.")

# ------------------------------------------------------------------------------
# 5. Walk-Forward Consensus Backtest (raw env: unscaled rupee P&L, no VecNormalize)
# ------------------------------------------------------------------------------
# norm_obs=False during training, so the agents consume raw env observations directly.
test_env = ConstrainedTradingEnv(df=test_df, **ENV_KWARGS)
state, info = test_env.reset(seed=42)
print(f"\n[Backtest] Continuous ensemble walk-forward over {len(test_df):,} candles...")

portfolio_values = [float(info["equity"])]
timestamps = [test_df.index[0]]
step_returns = []
trade_pnls = []
stop_loss_hits = 0
done = False

while not done:
    action_ppo, _ = ppo_model.predict(state, deterministic=True)
    action_a2c, _ = a2c_model.predict(state, deterministic=True)

    # Consensus voting: trade only when both agents agree on direction
    if action_ppo[0] > 0.1 and action_a2c[0] > 0.1:
        final_action = np.array([(action_ppo[0] + action_a2c[0]) / 2.0], dtype=np.float32)
    elif action_ppo[0] < -0.1 and action_a2c[0] < -0.1:
        final_action = np.array([(action_ppo[0] + action_a2c[0]) / 2.0], dtype=np.float32)
    else:
        final_action = np.array([0.0], dtype=np.float32)

    state, reward, terminated, truncated, step_info = test_env.step(final_action)
    done = terminated or truncated

    portfolio_values.append(float(step_info["equity"]))
    timestamps.append(test_df.index[step_info["step"]])
    step_returns.append(float(step_info["step_profit"]))
    trade_pnls.extend(step_info["closed_trade_pnls"])
    stop_loss_hits += int(step_info["stop_loss_hit"])

print(f"Backtest completed: {len(step_returns):,} steps, {step_info['closed_trades']} closed trades, "
      f"{stop_loss_hits} stop-loss exits.")

# ------------------------------------------------------------------------------
# 6. Institutional Performance Metrics (trade-level win rate & profit factor)
# ------------------------------------------------------------------------------
metrics = calculate_financial_metrics(
    portfolio_values=portfolio_values,
    step_returns=step_returns,
    total_trades=int(step_info["closed_trades"]),
    profitable_trades=int(step_info["winning_trades"]),
    initial_balance=float(test_env.initial_balance),
    final_balance=float(step_info["equity"]),
    total_fees_paid=float(step_info["total_fees_paid"]),
    bars_per_day=BARS_PER_DAY,
)
gross_wins = sum(p for p in trade_pnls if p > 0.0)
gross_losses = abs(sum(p for p in trade_pnls if p < 0.0))
metrics["profit_factor"] = (gross_wins / gross_losses) if gross_losses > 0.0 else (float("inf") if gross_wins > 0.0 else 0.0)
metrics["stop_loss_exits"] = stop_loss_hits
metrics["ticker"] = TICKER
metrics["test_start"] = str(test_df.index[0])
metrics["test_end"] = str(test_df.index[-1])

backtest_df = pd.DataFrame({
    "timestamp": timestamps,
    "portfolio_value": portfolio_values,
    "step_return": [0.0] + step_returns,
})
backtest_csv_path = os.path.join(RESULTS_PATH, "backtest_portfolio_history.csv")
backtest_df.to_csv(backtest_csv_path, index=False)

metrics_json_path = os.path.join(RESULTS_PATH, "evaluation_metrics.json")
with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("\n" + "=" * 65)
print("      ENSEMBLE (PPO + A2C) INSTITUTIONAL PERFORMANCE REPORT    ")
print("=" * 65)
print(f" Ticker / Test Window:     {TICKER} | {metrics['test_start'][:10]} -> {metrics['test_end'][:10]}")
print(f" Initial Balance:          ₹{metrics['initial_balance']:,.2f}")
print(f" Final Compounded Balance: ₹{metrics['final_balance']:,.2f}")
print(f" Net Profit (After Fees):  ₹{metrics['net_profit']:+,.2f} ({metrics['net_return_pct']:+.2f}%)")
print(f" Gross Profit (Pre-Fees):  ₹{metrics['gross_profit']:+,.2f} ({metrics['gross_return_pct']:+.2f}%)")
print(f" Total Brokerage & STT:    ₹{metrics['total_fees_paid']:,.2f}")
print(f" Closed Trades:            {metrics['total_trades']:,} (Win Rate: {metrics['win_rate_pct']:.1f}%, Stop-Loss Exits: {stop_loss_hits})")
print(f" Maximum Drawdown:         {metrics['max_drawdown_pct']:.2f}%")
print(f" Sharpe Ratio (Ann.):      {metrics['sharpe_ratio']:.4f}")
print(f" Sortino Ratio (Ann.):     {metrics['sortino_ratio']:.4f}")
pf = metrics["profit_factor"]
print(f" Profit Factor (Trades):   {pf:.2f}" if np.isfinite(pf) else " Profit Factor (Trades):   N/A (No Losing Trades)")
print("=" * 65)
print(f"Results CSV: {backtest_csv_path}")
print(f"Metrics:     {metrics_json_path}")

# ------------------------------------------------------------------------------
# 7. Equity Curve Visualization
# ------------------------------------------------------------------------------
plt.figure(figsize=(12, 5))
plt.plot(portfolio_values, color="#2563EB", linewidth=1.8, label="Ensemble Portfolio Equity (₹)")
plt.axhline(test_env.initial_balance, color="gray", linestyle="--", alpha=0.7, label="Initial Capital")
plt.title(f"ATU Multi-Agent Ensemble Out-of-Sample Equity Curve ({TICKER})", fontsize=13)
plt.xlabel("Intraday 5-Min Steps", fontsize=11)
plt.ylabel("Portfolio Balance (₹)", fontsize=11)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
equity_curve_path = os.path.join(RESULTS_PATH, "equity_curve.png")
plt.savefig(equity_curve_path, dpi=150)
plt.show()
print(f"Saved equity curve plot to: {equity_curve_path}")